In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("DPA-Part1") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()


1. Ingestion

Read ListenBrainz to explore the schema

In [3]:
temp = spark.read.parquet(
    "../data/raw/mb/listenbrainz-spark-dump-2702-20261006-000002-incremental/",
    "../data/raw/mb/listenbrainz-spark-dump-2704-20261007-000002-incremental/",
    "../data/raw/mb/listenbrainz-spark-dump-2706-20261008-000002-incremental/"
)


Print the schema of the raw ListenBrainz data

In [4]:
temp.printSchema()

root
 |-- listened_at: timestamp (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- recording_msid: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- artist_credit_id: long (nullable = true)
 |-- release_name: string (nullable = true)
 |-- release_mbid: string (nullable = true)
 |-- recording_name: string (nullable = true)
 |-- recording_mbid: string (nullable = true)
 |-- artist_credit_mbids: array (nullable = true)
 |    |-- element: string (containsNull = true)



Load ListenBrainz with explicit schema

In [5]:
from pyspark.sql.types import *

lb_schema = StructType([
    StructField("listened_at", TimestampType()),
    StructField("created", TimestampType()),
    StructField("user_id", LongType()),
    StructField("recording_msid", StringType()),
    StructField("artist_name", StringType()),
    StructField("artist_credit_id", LongType()),
    StructField("release_name", StringType()),
    StructField("release_mbid", StringType()),
    StructField("recording_name", StringType()),
    StructField("recording_mbid", StringType()),
    StructField("artist_credit_mbids", ArrayType(StringType())),
])

listens = spark.read.schema(lb_schema).parquet(
    "../data/raw/mb/listenbrainz-spark-dump-2702-20261006-000002-incremental/*.parquet",
    "../data/raw/mb/listenbrainz-spark-dump-2704-20261007-000002-incremental/*.parquet",
    "../data/raw/mb/listenbrainz-spark-dump-2706-20261008-000002-incremental/*.parquet"
)

print("Total listens:", listens.count())


Total listens: 11459693


Preview the listens data

In [6]:
listens.show(3, truncate=True, vertical=True)

-RECORD 0-----------------------------------
 listened_at         | 2026-10-05 23:58:15  
 created             | 2026-10-06 00:00:... 
 user_id             | 138729               
 recording_msid      | 237f105a-fae0-472... 
 artist_name         | Foo Fighters         
 artist_credit_id    | 176                  
 release_name        | In Your Honor        
 release_mbid        | 85f68702-8f92-3be... 
 recording_name      | Friend of a Friend   
 recording_mbid      | 11342c4c-f8db-474... 
 artist_credit_mbids | [67f66c07-6e61-40... 
-RECORD 1-----------------------------------
 listened_at         | 2026-10-05 23:59:52  
 created             | 2026-10-06 00:00:... 
 user_id             | 96459                
 recording_msid      | c2269c90-c760-48f... 
 artist_name         | Valve Studio Orch... 
 artist_credit_id    | 2722686              
 release_name        | Fight Songs: The ... 
 release_mbid        | 6d64ec07-bfa1-449... 
 recording_name      | Robots!              
 recording

Peek at raw MusicBrainz artist file

In [7]:
raw_artist = spark.read.text("../data/raw/mb/mbdump/artist")
raw_artist.show(3, truncate=False, vertical=True)


-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 2252039\tfadeb38c-833f-40bc-9d8c-a6383b38b1be\tДоктор Сатана\tДоктор Сатана\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\t0\t2021-11-23 07:08:52.479537+00\tf\t\N\t\N 
-RECORD 1---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 371203\t49add228-eac5-4de8-836c-d75cde7369c3\tPete Moutso\tMoutso, Pete\t\N\t\N\t\N\t\N\t\N\t\N\t1\t\N\t\N\t\t0\t\N\tf\t\N\t\N                                 
-RECORD 2---------------------------------------------------------------------------------------------------------------------------------------------------------------
 value | 3087346\tdfdce491-133d-4e9f-9e48-795587e181b0\tUNlT\tUNlT\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\N\t\t0\t2025-09-14 23:53:28.004798+00\tf\t\N\t\N      

Load artist with explicit schema

In [8]:
artist_schema = StructType([
    StructField("id", IntegerType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("sort_name", StringType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("type", IntegerType()),
    StructField("area", IntegerType()),
    StructField("gender", IntegerType()),
    StructField("comment", StringType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", StringType()),
    StructField("ended", StringType()),
    StructField("begin_area", IntegerType()),
    StructField("end_area", IntegerType()),
])

artist = spark.read.schema(artist_schema) \
    .option("sep", "\t") \
    .option("nullValue", "\\N") \
    .csv("../data/raw/mb/mbdump/artist")

print("Total artists:", artist.count())
artist.show(3, vertical=True, truncate=True)


Total artists: 3015712
-RECORD 0--------------------------------
 id               | 2252039              
 gid              | fadeb38c-833f-40b... 
 name             | Доктор Сатана        
 sort_name        | Доктор Сатана        
 begin_date_year  | NULL                 
 begin_date_month | NULL                 
 begin_date_day   | NULL                 
 end_date_year    | NULL                 
 end_date_month   | NULL                 
 end_date_day     | NULL                 
 type             | NULL                 
 area             | NULL                 
 gender           | NULL                 
 comment          | NULL                 
 edits_pending    | 0                    
 last_updated     | 2021-11-23 07:08:... 
 ended            | f                    
 begin_area       | NULL                 
 end_area         | NULL                 
-RECORD 1--------------------------------
 id               | 371203               
 gid              | 49add228-eac5-4de... 
 name      

Peek at raw MusicBrainz area file

In [9]:
raw_area = spark.read.text("../data/raw/mb/mbdump/area")
raw_area.show(3, truncate=False, vertical=True)


-RECORD 0-----------------------------------------------------------------------------------------------------------------------
 value | 15449\t2913ad77-cec8-4d2f-98d3-d4aa46ab73bc\tGreccio\t4\t0\t2013-07-21 22:47:57.660809+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t 
-RECORD 1-----------------------------------------------------------------------------------------------------------------------
 value | 38\t71bbafaa-e825-3e15-8ca9-017dcad1748b\tCanada\t1\t0\t2013-05-27 13:15:52.179105+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t     
-RECORD 2-----------------------------------------------------------------------------------------------------------------------
 value | 43\t82d5f4d6-aed4-3ff5-81d1-5363ac6e97a7\tChile\t1\t0\t2013-05-27 12:52:17.320228+00\t\N\t\N\t\N\t\N\t\N\t\N\tf\t      
only showing top 3 rows



Load area with explicit schema

In [10]:
area_schema = StructType([
    StructField("id", IntegerType()),
    StructField("gid", StringType()),
    StructField("name", StringType()),
    StructField("type", IntegerType()),
    StructField("edits_pending", IntegerType()),
    StructField("last_updated", StringType()),
    StructField("begin_date_year", IntegerType()),
    StructField("begin_date_month", IntegerType()),
    StructField("begin_date_day", IntegerType()),
    StructField("end_date_year", IntegerType()),
    StructField("end_date_month", IntegerType()),
    StructField("end_date_day", IntegerType()),
    StructField("ended", StringType()),
    StructField("comment", StringType()),
])

area = spark.read.schema(area_schema) \
    .option("sep", "\t") \
    .option("nullValue", "\\N") \
    .csv("../data/raw/mb/mbdump/area")

print("Total areas:", area.count())
area.show(3, vertical=True, truncate=True)


Total areas: 120108
-RECORD 0--------------------------------
 id               | 15449                
 gid              | 2913ad77-cec8-4d2... 
 name             | Greccio              
 type             | 4                    
 edits_pending    | 0                    
 last_updated     | 2013-07-21 22:47:... 
 begin_date_year  | NULL                 
 begin_date_month | NULL                 
 begin_date_day   | NULL                 
 end_date_year    | NULL                 
 end_date_month   | NULL                 
 end_date_day     | NULL                 
 ended            | f                    
 comment          | NULL                 
-RECORD 1--------------------------------
 id               | 38                   
 gid              | 71bbafaa-e825-3e1... 
 name             | Canada               
 type             | 1                    
 edits_pending    | 0                    
 last_updated     | 2013-05-27 13:15:... 
 begin_date_year  | NULL                 
 begin_date_mo

Write to Parquet

In [11]:
listens.write.mode("overwrite").parquet("../data/processed/listens")
artist.write.mode("overwrite").parquet("../data/processed/artist")
area.write.mode("overwrite").parquet("../data/processed/area")

print("Parquet tables written successfully")


Parquet tables written successfully


2. Cleaning 

null check

In [12]:
from pyspark.sql.functions import col, count, when

def null_counts(df, name):
    print(f"\n=== {name} ===")
    df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(vertical=True)

null_counts(listens, "listens")
null_counts(artist, "artist")
null_counts(area, "area")



=== listens ===
-RECORD 0----------------------
 listened_at         | 0       
 created             | 0       
 user_id             | 0       
 recording_msid      | 0       
 artist_name         | 0       
 artist_credit_id    | 2146861 
 release_name        | 297380  
 release_mbid        | 2148400 
 recording_name      | 0       
 recording_mbid      | 2100611 
 artist_credit_mbids | 2001475 


=== artist ===
-RECORD 0-------------------
 id               | 0       
 gid              | 0       
 name             | 0       
 sort_name        | 2       
 begin_date_year  | 2409523 
 begin_date_month | 2674124 
 begin_date_day   | 2696118 
 end_date_year    | 2859059 
 end_date_month   | 2911709 
 end_date_day     | 2917177 
 type             | 684516  
 area             | 1518240 
 gender           | 1736414 
 comment          | 1951921 
 edits_pending    | 9       
 last_updated     | 136245  
 ended            | 9       
 begin_area       | 2515563 
 end_area         | 2938873 




delete bad rows and delete useless columns

In [13]:
listens_clean = listens.dropna(subset=["artist_credit_id"]) \
    .select("listened_at", "user_id", "artist_credit_id", "artist_name", "recording_mbid")

artist_clean = artist.dropna(subset=["area"]) \
    .select("id", "name", "area")

area_clean = area.select("id", "name")

print("listens_clean:", listens_clean.count())
print("artist_clean:", artist_clean.count())
print("area_clean:", area_clean.count())


listens_clean: 9312832
artist_clean: 1497472
area_clean: 120108


write the cleaned tables

In [14]:
listens_clean.write.mode("overwrite").parquet("../data/processed/listens_clean")
artist_clean.write.mode("overwrite").parquet("../data/processed/artist_clean")
area_clean.write.mode("overwrite").parquet("../data/processed/area_clean")

print("Cleaned tables written.")


Cleaned tables written.


3.Spark SQL analysis